In [51]:
import os
from dotenv import load_dotenv

# MODEL = "gemini-pro"
MODEL = "gemini-1.5-flash"

In [52]:
load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

In [53]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/embedding-001",
    google_api_key= GEMINI_API_KEY
)

In [11]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("resources/humourous_short_stories.pdf")
pages = loader.load_and_split()
pages

[Document(metadata={'source': 'resources/humourous_short_stories.pdf', 'page': 0}, page_content='THE BEST AMERICAN\nHUMOROUS SHORT\nSTORIES\nby Various Authors\nStyled by LimpidSoft'),
 Document(metadata={'source': 'resources/humourous_short_stories.pdf', 'page': 1}, page_content='Contents\nINTRODUCTION 4\nACKNOWLEDGMENTS 46\nTHE LITTLE FRENCHMAN AND HIS WATER\nLOTS 49\nTHE ANGEL OF THE ODD 61\nTHE SCHOOLMASTER’S PROGRESS 79\nTHE WATKINSON EVENING 106\n2'),
 Document(metadata={'source': 'resources/humourous_short_stories.pdf', 'page': 2}, page_content='CONTENTS\nTITBOTTOM’S SPECTACLES 138\nMY DOUBLE; AND HOW HE UNDID ME 178\nA VISIT TO THE ASYLUM FOR AGED AND DE-\nCAYED PUNSTERS 210\nTHE CELEBRATED JUMPING FROG OF\nCALAVERAS COUNTY 225\nELDER BROWN’S BACKSLIDE 237\nI . . . . . . . . . . . . . . . . . . . . . . . . . . . . 238\nII . . . . . . . . . . . . . . . . . . . . . . . . . . . 245\nIII . . . . . . . . . . . . . . . . . . . . . . . . . . . 261\nIV . . . . . . . . . . . . . . . . .

In [54]:
from langchain.prompts import PromptTemplate

template = """
You are a creative writer. Below is a complete book that should serve as your inspiration for writing style and tone.
Use the full context as inspiration. Write a new piece that captures a similar essence while being original.
It should be about 500 words. Must use the 10 words provided in the question. Not mandatory to use the same order.

Context: {context}

Requirements: {question}

Write a creative piece following the above requirements while matching the writing style from the context.
Give the story a suitable and engaging name as well.
"""


prompt = PromptTemplate.from_template(template)
print(prompt.format(context="Here is some context", question="Here is a question"))


You are a creative writer. Below is a complete book that should serve as your inspiration for writing style and tone.
Use the full context as inspiration. Write a new piece that captures a similar essence while being original.
It should be about 500 words. Must use the 10 words provided in the question. Not mandatory to use the same order.

Context: Here is some context

Requirements: Here is a question

Write a creative piece following the above requirements while matching the writing style from the context.
Give the story a suitable and engaging name as well.



In [55]:
from langchain_community.vectorstores import DocArrayInMemorySearch

vectorstore = DocArrayInMemorySearch.from_documents(
    pages, 
    embedding = embeddings
)

In [56]:
vector = embeddings.embed_query("How are you")
len(vector)

768

In [19]:
retriever = vectorstore.as_retriever()
retriever.invoke("Inmates who have lost their faculties and cannot any longer make Puns shall be permitted to repeat such as may be selected for them by the Chaplain out of the work of Mr. Joseph Miller")

[Document(metadata={'source': 'resources/humourous_short_stories.pdf', 'page': 215}, page_content='A VISIT TO THE ASYLUM FOR AGED AND\nDECAYED PUNSTERS\n6. At ten o’clock the gas will be turned off, and no fur-\nther Puns, Conundrums, or other play on words will be\nallowed to be uttered, or to be uttered aloud.\n9. Inmates who have lost their faculties and cannot any\nlonger make Puns shall be permitted to repeat such as\nmay be selected for them by the Chaplain out of the work\nof Mr. Joseph Miller.\n10. Violent and unmanageable Punsters, who inter-\nrupt others when engaged in conversation, with Puns or\nattempts at the same, shall be deprived of their Joseph\nMillers, and, if necessary, placed in solitary conﬁnement.\nSECT. III. OF DEPORTMENT AT MEALS.\n4. No Inmate shall make any Pun, or attempt at the\nsame, until the Blessing has been asked and the company\nare decently seated.\n7. Certain Puns having been placed on the Index Ex-\npurgatorius of the Institution, no Inmate shall 

In [57]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model=MODEL,
    google_api_key=GEMINI_API_KEY                    
)

response = llm.invoke("Write a 5 line poem on AI")

print(response.content)

A mind of code, a silent hum,
Learning patterns, overcome.
With data fed, it grows and thrives,
A future shaped, where knowledge lives.
But human touch, still it requires.



In [20]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

chain = llm | parser
chain.invoke("Tell me a joke")

'Why did the scarecrow win an award?\n\nBecause he was outstanding in his field!'

In [58]:
from langchain.chains import LLMChain
from operator import itemgetter

# selected_pages = pages[50:107]  # Using 107 as end index since slicing is exclusive
# all_text = "\n".join([page.page_content for page in selected_pages])

all_text = "\n".join([page.page_content for page in pages])

chain = (
    {
        "context": lambda x: all_text,
        "question": itemgetter("question"),
    }
    | prompt
    | llm
    | parser
)


In [59]:
question = '''
[pugnacious, profligate, dereliction, extol, burnish, impending, stigmatize, delineate, refractory, disinter]
'''

In [60]:
answer = chain.invoke({"question": question})
print(answer)

**The Case of the Gilded Lily and the Refractory Rooster**

Professor Thaddeus P. Quibble, a man whose pugnacious spirit was only surpassed by his encyclopedic knowledge of obscure historical anecdotes, found himself, much against his will, embroiled in a most peculiar case.  It concerned Miss Clementine “Clem” Buttercup, a young lady whose beauty, like a burnished coin, shone brightly, yet whose reputation, thanks to a certain profligate, had been unfairly stigmatized.  The profligate in question was none other than Bartholomew “Barty” Cockleburr, a pillar of the local church and a man whose piety was as ostentatious as his dereliction of duty in this matter.

Professor Quibble, a man who extolled the virtues of reason and logic above all else, found himself strangely drawn to Clem’s plight.  Her story, as she delineated it in her hesitant, almost childlike manner, was a tapestry woven with threads of seemingly innocuous incidents. There were stolen glances across the hymnals during p

## Image Generation

In [50]:
import google.generativeai as genai

genai.configure(api_key=GEMINI_API_KEY)

# imagen = genai.ImageGenerationModel("imagen-3.0-generate-001")

from vertexai.preview.generative_models import GenerativeModel

image_model = GenerativeModel(
  "imagen"
)


result = image_model.generate_images(
    prompt=answer,
    number_of_images=4,
    safety_filter_level="block_only_high",
    person_generation="allow_adult",
    aspect_ratio="3:4",
    negative_prompt="Outside",
)

for image in result.images:
  print(image)

# Open and display the image using your local operating system.
for image in result.images:
  image._pil_image.show()

TypeError: _GenerativeModel.__init__() takes 2 positional arguments but 3 were given

In [1]:
import os
from dotenv import load_dotenv

In [2]:
load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

In [3]:
from google import genai
from google.genai.types import HttpOptions

client = genai.Client(http_options=HttpOptions(api_version="v1"))
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="How does AI work?",
)
print(response.text)
# Example response:
# Okay, let's break down how AI works. It's a broad field, so I'll focus on the ...
#
# Here's a simplified overview:
# ...

AI, at its core, works by **learning from data** rather than being explicitly programmed for every single task. Think of it like teaching a child: instead of giving them a rulebook for every situation, you show them examples, give them feedback, and they gradually learn to generalize and make their own decisions.

Here's a breakdown of the key components and processes:

### The Core Idea: Learning from Data

Traditional software follows a strict set of instructions written by a programmer. If the situation isn't covered by the rules, it fails. AI, especially Machine Learning (a subset of AI), flips this:

1.  **You feed it a lot of data.** This data can be images, text, numbers, audio, etc.
2.  **The AI identifies patterns and relationships** within that data.
3.  **It builds a "model"** based on these patterns. This model is essentially a complex mathematical representation of what it has learned.
4.  **It then uses this model to make predictions, decisions, or perform tasks** on new,

In [4]:
from google import genai
from google.genai.types import GenerateContentConfig, Modality
from PIL import Image
from io import BytesIO

client = genai.Client()

response = client.models.generate_content(
    model="gemini-2.0-flash-preview-image-generation",
    contents=(
        "Generate an image of the Eiffel tower with fireworks in the background."
    ),
    config=GenerateContentConfig(response_modalities=[Modality.TEXT, Modality.IMAGE]),
)
for part in response.candidates[0].content.parts:
    if part.text:
        print(part.text)
    elif part.inline_data:
        image = Image.open(BytesIO((part.inline_data.data)))
        image.save("example-image.png")
# Example response:
#   A beautiful photograph captures the iconic Eiffel Tower in Paris, France,
#   against a backdrop of a vibrant and dynamic fireworks display. The tower itself...

I will generate an image of the Eiffel Tower at night, with a vibrant and colorful fireworks display illuminating the sky behind it, creating a festive and celebratory atmosphere.




In [8]:
response.candidates[0].content.parts[1].inline_data

Blob(
  data=b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x04\x00\x00\x00\x01\xfe\x08\x02\x00\x00\x00\x8d\xdf\xeb\x1a\x00\x00\x00\x89zTXtRaw profile type iptc\x00\x00\x08\x99M\x8c1\x0e\x021\x0c\x04\xfb\xbc\xe2\x9e\x908\xeb\xb5]S\xd1Q\xf0\x81\xbb\\"!!\x81\xf8\x7fA...',
  mime_type='image/png'
)